### RAG Pipelines: data ingestoin to vector DB pipeline

In [1]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
import os
from pathlib import Path


C:\Users\sidsa\AppData\Local\Temp\ipykernel_22208\2958759400.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
def process_all_pdfs(pdf_dir):
    all_documents = []
    pdf_dir = Path(pdf_dir)
    pdf_files = list(pdf_dir.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in {pdf_dir}")
    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            # add source information to the metadata
            for document in documents:
                document.metadata["file_name"] = pdf_file.name
                document.metadata["source"] = str(pdf_file)
                document.metadata["file_type"] =  'pdf'
            all_documents.extend(documents)
            print(f"Added {len(documents)} pages from {pdf_file}")
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

all_documents = process_all_pdfs("../data/pdf")
print(f"Found {len(all_documents)} documents")



Found 2 PDF files in ..\data\pdf
Processing ..\data\pdf\MicroServices_Concepts.pdf
Added 18 pages from ..\data\pdf\MicroServices_Concepts.pdf
Processing ..\data\pdf\SiddhantSanchetik_Resume.pdf
Added 2 pages from ..\data\pdf\SiddhantSanchetik_Resume.pdf
Total documents loaded: 20
Found 20 documents


In [3]:
all_documents

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\MicroServices_Concepts.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1', 'file_name': 'MicroServices_Concepts.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\MicroServices_Concepts.pdf', 'total_pages': 18, 'page': 1, 'page_label': '2', 'file_name': 'MicroServices_Concepts.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\MicroServices_Concepts.pdf', 'total_pages': 18, 'page': 2, 'page_label': '3', 'file_name': 'MicroServices_Concepts.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\MicroServices_Concepts.pdf', 'total_pages': 18, 'page': 3, 'page_label': '4', 'file_name

In [4]:
### Text Splitter
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len, separators=["\n\n", "\n", " ", ""])

    split_docs = text_splitter.split_documents(all_documents)
    print(f"Split {len(split_docs)} documents into {len(split_docs)} chunks")

    # show example of a chunk
    if split_docs:
        print(f"Example of a chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs


In [5]:
chunks = split_documents(all_documents)
chunks

Split 9 documents into 9 chunks
Example of a chunk:
Content: SIDDHANT K. SANCHETI
Software Engineer | Full Stack, AI/ML & Distributed Systems
(669) 319-8812 • siddhantsanchetik@gmail.com • linkedin.com/in/siddhant-sancheti • siddhantsancheti.com
Summary
Softwar...
Metadata: {'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-10-23T23:14:26+00:00', 'author': '', 'keywords': '', 'moddate': '2026-10-23T23:14:26+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\SiddhantSanchetik_Resume.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'file_name': 'SiddhantSanchetik_Resume.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-10-23T23:14:26+00:00', 'author': '', 'keywords': '', 'moddate': '2026-10-23T23:14:26+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\SiddhantSanchetik_Resume.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'file_name': 'SiddhantSanchetik_Resume.pdf', 'file_type': 'pdf'}, page_content='SIDDHANT K. SANCHETI\nSoftware Engineer | Full Stack, AI/ML & Distributed Systems\n(669) 319-8812 • siddhantsanchetik@gmail.com • linkedin.com/in/siddhant-sancheti • siddhantsancheti.com\nSummary\nSoftware Engineer with 3+ years designing distributed systems, backend services, full-stack products, and production multi-agent AI, backed by evals,\nreliability guardrails, and measurable impact. At AWS, co-designed a multi-agent orchestration system adopted 

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


d:\Projects\RAG-crash-course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading model {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise e
    
    def generate_embeddings(self, texts):
        if not self.model:
            raise ValueError("Model not loaded. Please load the model first.")
        print(f"Generating embeddings for {len(texts)} texts")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Embeddings generated successfully. Shape: {embeddings.shape}")
        return embeddings
    
## initialize embedding manager
embedding_manager = EmbeddingManager()

# embeddings = embedding_manager.generate_embeddings(chunks)
embedding_manager

# ## initialize chroma client
# client = chromadb.Client()
# client

# ## initialize chroma collection
# collection = client.create_collection("pdf_embeddings")
# collection

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8342.12it/s]


Loading model all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3662.90it/s]


Model loaded successfully. Embedding dimension: 384


In [8]:
###  VectorStore
#### manages document embeddings in a chroma db store
class VectorStore:
        
    def __init__(self, collection_name="pdf_documents", persistent_dir="../data/vector_store"):
        self.collection_name = collection_name
        self.persistent_dir = persistent_dir
        self.client = None
        self.collection = None
        self.initialize_store()


    def initialize_store(self):
        """Initialize the chroma client and collection"""
        try:
            # Create persistent Chroma DB client
            os.makedirs(self.persistent_dir, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persistent_dir)
            print(f"Chroma DB client initialized successfully. Persistent directory: {self.persistent_dir}")

            # Cosine distance matches similarity_score = 1 - distance
            collection_metadata = {
                "description": "PDF documents embeddings for RAG",
                "hnsw:space": "cosine",
            }
            existing = None
            try:
                existing = self.client.get_collection(self.collection_name)
            except Exception:
                existing = None

            if existing is not None and (existing.metadata or {}).get("hnsw:space") != "cosine":
                stored = existing.get(include=["documents", "embeddings", "metadatas"])
                self.client.delete_collection(self.collection_name)
                self.collection = self.client.create_collection(
                    self.collection_name,
                    metadata=collection_metadata,
                )
                if stored["ids"]:
                    self.collection.add(
                        ids=stored["ids"],
                        documents=stored["documents"],
                        metadatas=stored["metadatas"],
                        embeddings=stored["embeddings"],
                    )
                print("Recreated collection with cosine distance and restored existing documents")
            else:
                self.collection = self.client.get_or_create_collection(
                    self.collection_name,
                    metadata=collection_metadata,
                )
            print(f"Vector store initialized successfully. Collection name: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing Chroma DB client: {e}")
            raise e

    def add_documents(self, documents:List[Any], embeddings:np.ndarray):
        """Add documents and embeddings to the vector store"""
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must be the same")

        print(f"Adding {len(documents)} documents to the vector store")

        # Prepare data for chroma db
        ids = []
        metadatas = []
        documents_text  = []
        embeddings_list = []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #Generate a unique id for the document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            documents.append(doc.page_content)


            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        if self.collection is None:
            raise RuntimeError("Vector store collection is not initialized")
        try:
            self.collection.add(
                ids=ids,
                documents=documents_text,
                metadatas=metadatas,
                embeddings=embeddings_list
            )
            print(f"Documents added to the vector store successfully. Number of documents: {len(documents)}")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise e

vector_store = VectorStore()
vector_store

Chroma DB client initialized successfully. Persistent directory: ../data/vector_store
Vector store initialized successfully. Collection name: pdf_documents
Existing documents in collection: 108


In [9]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-10-23T23:14:26+00:00', 'author': '', 'keywords': '', 'moddate': '2026-10-23T23:14:26+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\SiddhantSanchetik_Resume.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'file_name': 'SiddhantSanchetik_Resume.pdf', 'file_type': 'pdf'}, page_content='SIDDHANT K. SANCHETI\nSoftware Engineer | Full Stack, AI/ML & Distributed Systems\n(669) 319-8812 • siddhantsanchetik@gmail.com • linkedin.com/in/siddhant-sancheti • siddhantsancheti.com\nSummary\nSoftware Engineer with 3+ years designing distributed systems, backend services, full-stack products, and production multi-agent AI, backed by evals,\nreliability guardrails, and measurable impact. At AWS, co-designed a multi-agent orchestration system adopted 

In [10]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]
### Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)


vector_store.add_documents(chunks, embeddings)

Generating embeddings for 9 texts


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.25it/s]

Embeddings generated successfully. Shape: (9, 384)
Adding 9 documents to the vector store
Documents added to the vector store successfully. Number of documents: 18
Total documents in collection: 117


In [11]:
## Retriever Pipeline from VectorStore

class RAGRetriever:
    """Handles query processing and retrieval from vector store"""
    def __init__(self, vector_store:VectorStore, embedding_manager:EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve relevant documents for a query"""
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        client = self.vector_store.client
        if client is None:
            raise RuntimeError("Vector store client is not initialized")
        collection = client.get_collection(self.vector_store.collection_name)
        self.vector_store.collection = collection
        try:
            results = collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            retrieved_docs = []
            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                distances = results["distances"][0]
                metadatas = results["metadatas"][0]
                ids = results["ids"][0]
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Cosine distance: similarity is 1 - distance
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1,
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vector_store, embedding_manager)
results = rag_retriever.retrieve("Who is Siddhant Sancheti?")
rag_retriever.retrieve("What is the main topic of the document?", top_k=10, score_threshold=0.5)
results

Retrieving documents for query: 'Who is Siddhant Sancheti?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 106.09it/s]


Embeddings generated successfully. Shape: (1, 384)
Retrieved 5 documents (after filtering)
Retrieving documents for query: 'What is the main topic of the document?'
Top K: 10, Score threshold: 0.5
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 147.02it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 0 documents (after filtering)


[{'id': 'doc_92efe102_0',
  'content': 'SIDDHANT K. SANCHETI\nSoftware Engineer | Full Stack, AI/ML & Distributed Systems\n(669) 319-8812 • siddhantsanchetik@gmail.com • linkedin.com/in/siddhant-sancheti • siddhantsancheti.com\nSummary\nSoftware Engineer with 3+ years designing distributed systems, backend services, full-stack products, and production multi-agent AI, backed by evals,\nreliability guardrails, and measurable impact. At AWS, co-designed a multi-agent orchestration system adopted by 20+ teams, built self-service fleet\ncapacity APIs used by 50+ teams across 5,000+ EC2 hosts, and led a zero-downtime IAM migration across 110+ services in 17 regions. Previously shipped\nRAG-powered and full-stack products serving 5,000 to 100,000+ users.\nWork Experience\nAmazon Web Services May 2025 – Present\nSoftware Development Engineer Seattle, WA\n• Co-designed a DAG-based Python/Go multi-agent orchestration system adopted by 20+ teams, cutting delivery cycles across 195+ packages from 

### Simple RAG Pipeline - VectorDB to LLM output generation

In [12]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [13]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [14]:
class GroqLLM:
    def __init__(self, model_name: str = "openai/gpt-oss-120b", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (openai/gpt-oss-120b, openai/gpt-oss-20b, qwen/qwen3.8-27b)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    


In [15]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: openai/gpt-oss-120b
Groq LLM initialized successfully!


In [16]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("What are the expertise of Siddhant Sancheti")

Retrieving documents for query: 'What are the expertise of Siddhant Sancheti'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 116.11it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_8ee9809f_0',
  'content': 'SIDDHANT K. SANCHETI\nSoftware Engineer | Full Stack, AI/ML & Distributed Systems\n(669) 319-8812 • siddhantsanchetik@gmail.com • linkedin.com/in/siddhant-sancheti • siddhantsancheti.com\nSummary\nSoftware Engineer with 3+ years designing distributed systems, backend services, full-stack products, and production multi-agent AI, backed by evals,\nreliability guardrails, and measurable impact. At AWS, co-designed a multi-agent orchestration system adopted by 20+ teams, built self-service fleet\ncapacity APIs used by 50+ teams across 5,000+ EC2 hosts, and led a zero-downtime IAM migration across 110+ services in 17 regions. Previously shipped\nRAG-powered and full-stack products serving 5,000 to 100,000+ users.\nWork Experience\nAmazon Web Services May 2025 – Present\nSoftware Development Engineer Seattle, WA\n• Co-designed a DAG-based Python/Go multi-agent orchestration system adopted by 20+ teams, cutting delivery cycles across 195+ packages from 

### Integration VectorDb context pipeline with LLM output

In [17]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key, model_name="openai/gpt-oss-120b", temperature=0.1, max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [18]:
answer = rag_simple("what is a load balancer", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'what is a load balancer'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 85.09it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 3 documents (after filtering)


A load balancer is a networking component (hardware or software) that distributes incoming traffic or requests across multiple servers or service instances. By spreading the workload, it improves application availability, fault tolerance, and performance, ensuring no single server becomes a bottleneck and that traffic can be rerouted automatically if a server fails.


### Enhanced RAG

In [19]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("difference between load balancers and gateway", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'difference between load balancers and gateway'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 164.08it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: **Load balancer**  
- **Purpose:** Distributes incoming network traffic across a pool of backend servers to improve availability, reliability, and scalability.  
- **Scope:** Primarily concerned with *traffic distribution* and health‑checking; operates at L4 (TCP/UDP) or L7 (HTTP) levels.  
- **Features:** Session‑persistence, SSL termination, simple routing (round‑robin, least‑connections, etc.).  

**Gateway (API gateway / service gateway)**  
- **Purpose:** Acts as a single entry point for client requests to a set of services or APIs, handling not just routing but also cross‑cutting concerns.  
- **Scope:** Provides *request mediation*—authentication/authorization, rate limiting, request/response transformation, caching, logging, and sometimes load‑balancing.  
- **Features:** API versioning, schema validation, protocol translation (e.g., HTTP ↔ gRPC), service discovery, and policy enforcement.  

**Key difference:** A load balancer’s core job is to spread traffic evenly acr

In [20]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what are microservices concepts", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what are microservices concepts'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 174.73it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Skills
Programming Languages: Python, Java, Go, Ruby, TypeScript, JavaScript, SQL, Bash, Shell
Backend & Infrastructure: Spring Boot, Guice, Node.js, Express.js, Flask, FastAPI, Django, React, Redux, Bootstrap, REST APIs, Remote Procedure Call
(RPC), GraphQL, Amazon Web Services (AWS Lambda, EC2, S3, ECS, Bedrock, Route 53, CDK),

 Google Cloud Platform (GCP), Kubernetes, Docker, Terraform
Distributed Systems: Microservices, Event-Driven Architecture, Distributed Locking, Circuit Breakers, Concurrency, Asynchronous Programming, Apache
Kafka, Apache ZooKeeper, Apache Spark
Databases: PostgreSQL, Aurora, RDS, DynamoDB, Redshift, MongoDB, Firebase
Generative AI & Agentic Systems: Multi-Agent Orchestration, Large Language Models (LLMs), GPT-4, Retrieval-Augmented Generation (RAG), Re-ranking,
LLM Evals, LangChain, LangGraph, CrewAI, Pydantic AI, Model Context Protocol (MCP), A2A Protocol, OpenAI Agents SDK, Composio, StackOne, Mem0, Zep,
AWS AgentCore

Skills
Programming Languages: Python, Java, Go, Ruby, TypeScript, JavaScript, SQL, Bash, Shell
Backend & Infrastructure: Spring Boot, Guice, Node.js, Express.js, Flask, FastAPI, Django, React, Redux, Bootstrap, REST APIs, Remote Procedure Call
(RPC), GraphQL, Amazon Web Services (AWS Lambda, EC2, S3, ECS, Bedrock, Route 53, CDK), Google Cloud Platform (GCP), Kubernete